# Colab test · RL Workshop

Quick check that the workshop code runs in Google Colab. Run the cells top to bottom (**Runtime → Run all** or Shift+Enter).

This is your own copy: changes do not affect the original. To keep your work: **File → Save a copy in Drive**.

## 1. Setup
On Colab this downloads the workshop repository (agent code and data). Locally or in the browser lab it does nothing.

In [ ]:
import sys, os
if "google.colab" in sys.modules:
    if not os.path.exists("/content/rl-workshop"):
        !git clone -q --depth 1 https://github.com/jakob284/rl-workshop.git /content/rl-workshop
    %cd /content/rl-workshop/notebooks

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
root = next((p for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent] if (p / "asset_agent.py").exists()), None)
assert root is not None, "asset_agent.py not found: run the setup cell first"
sys.path.insert(0, str(root))
from asset_agent import PortfolioAgent, DEFAULTS
df = pd.read_csv(root / "data/historical_normalized_weekly_stocks_2018_2019.csv", parse_dates=["date"])
prices = df["AAPL"].to_numpy(dtype=float).round(8)
dates = df["date"].dt.strftime("%Y-%m-%d").tolist()
print(f"Loaded {len(prices)} weekly Apple prices, {dates[0]} to {dates[-1]}")

## 2. Train the Q-learning agent and evaluate it

In [ ]:
agent = PortfolioAgent(prices, dates, **DEFAULTS)
agent.step_many(len(agent.train_t) * agent.budget)
results = agent.evaluate()
display(pd.DataFrame([{"Strategy": r["name"], "Net total return": r["totalReturn"], "Max drawdown": r["maxDrawdown"]} for r in results]).round(4))
fig, ax = plt.subplots(figsize=(9, 4))
for r in results:
    ax.plot(r["wealth"], label=r["name"])
ax.set(title="Evaluation-period wealth (starts at 1.00)", xlabel="Weeks", ylabel="Wealth")
ax.legend(); ax.grid(alpha=.2); plt.show()

## 3. Your turn
Change the settings below and rerun. Try `gamma`, `cost_bps` or `risk`.

In [ ]:
my_settings = {**DEFAULTS, "gamma": 0.5, "cost_bps": 10}
mine = PortfolioAgent(prices, dates, **my_settings)
mine.step_many(len(mine.train_t) * mine.budget)
print("My net test return:", round(mine.evaluate()[0]["totalReturn"], 4))

## 4. Is PyTorch available?
Colab has PyTorch preinstalled (needed later for deep RL). Select **Runtime → Change runtime type → GPU** to get a free GPU.

In [ ]:
try:
    import torch
    print("PyTorch", torch.__version__, "| GPU available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed here (expected in the browser lab; use Colab for deep RL)")